In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --- Helper functies ---

def calculate_z(x, y, a, b, j):
    term1 = np.sin(y - np.sin(x) * a + 0.5 * np.pi)
    term2 = (j + np.sin(x - 0.5 * np.pi) * 2) *(1.0 / (j * (2 * np.pi) ** 0.5)) * np.exp(-0.5 * (y / j) ** 2)
    
    
    z = term1 * b * term2

    # Toepassen van de voorwaarden
    # {sin(x) * a - pi <= y <= sin((x)) * a + pi}
    condition_y_lower = np.sin(x) * a - np.pi
    condition_y_upper = np.sin(x) * a + np.pi
    



    
    # {-j + 2 <= z}
    condition_z_lower = -j + 2

    # Maak een masker voor de punten die niet aan de voorwaarden voldoen
    # Als een punt niet aan de voorwaarden voldoet, wordt de waarde ervan NaN (Not a Number)
    # zodat deze niet wordt geplot.
    mask = (y >= condition_y_lower) & \
           (y <= condition_y_upper) & \
           (z >= condition_z_lower)
    
    z_filtered = np.where(mask, z, 0)
    return z_filtered

# Functie om de minimale hoekfout te berekenen (komt overeen met self.err)
def calculate_error(current_th, target_th):
    return np.abs(((current_th - target_th + np.pi) % (2 * np.pi)) - np.pi)

# Helper functie voor 2D Gaussian
def gaussian_2d(val_x, val_y, mu_x, mu_y, sigma_x, sigma_y, rho, scale):
    # Zorg ervoor dat inputs arrays zijn, zelfs als ze scalair zijn
    val_x = np.atleast_1d(val_x)
    val_y = np.atleast_1d(val_y)
    mu = np.array([mu_x, mu_y])
    cov = np.array([[sigma_x**2, rho * sigma_x * sigma_y],
                    [rho * sigma_x * sigma_y, sigma_y**2]])
    inv_cov = np.linalg.inv(cov)
    det_cov = np.linalg.det(cov)
    diff_x = val_x - mu[0]
    diff_y = val_y - mu[1]
    exponent = -0.5 * (inv_cov[0,0] * diff_x**2 +
                        (inv_cov[0,1] + inv_cov[1,0]) * diff_x * diff_y +
                        inv_cov[1,1] * diff_y**2)

    return scale * (1.0 / (2 * np.pi * np.sqrt(det_cov))) * np.exp(exponent)




A_VALUE = 3.25  # Voor 'a' in de formule
B_VALUE = 0.7  # Voor 'b' in de formule
J_VALUE = 2  # Voor 'j' in de formule







def reward_function_for_plot(th_val, omega_val, th_ref_val, u_val):
    main_reward_component = gaussian_2d(calculate_error(th_val, np.pi), omega_val, 0, 0, 1, 1, 0.0, 2)
    bottom_punishment_component = gaussian_2d(calculate_error(th_val, 0), omega_val, 0, 0, 3, 3, 0.0, 40)
    bottom_punishment_component_2 = gaussian_2d(calculate_error(th_val, 0), omega_val, 0, 0, 1, 12, 0.0, 60)
    piek_reward_component = gaussian_2d(calculate_error(th_val, np.pi), omega_val, 0, 0, 0.07, 0.07, 0.0, 0.15)
    piek_test = gaussian_2d(calculate_error(th_val, np.pi), omega_val, 0, 0, 0.15, 0.15, 0.0, 0.5)
    Z_VALUES = calculate_z(th_val, omega_val, a=A_VALUE, b=B_VALUE, j=J_VALUE)
    
    control_input_penalty = 0.001 * u_val**2
    # Combineer alle componenten
    total_reward = (
        main_reward_component
        - bottom_punishment_component
        - control_input_penalty
        - bottom_punishment_component_2
         + Z_VALUES
        #  + piek_reward_component
         + piek_test
    )
    return total_reward

# --- Plotting gedeelte ---






# Definieer de ranges voor de assen van de plot
# th_val: De hoek van je systeem, van -pi tot pi (een volledige cirkel)
th_plot_range = np.linspace(-2 * np.pi, 2 * np.pi, 400)
# omega_val: De hoeksnelheid
omega_plot_range = np.linspace(-5, 5, 400) 

# Creëer de meshgrid voor de plotassen
TH_MESH, OMEGA_MESH = np.meshgrid(th_plot_range, omega_plot_range)

# Kies de vaste waarden voor th_ref en u voor DÉZE specifieke plot
# Je kunt deze waarden aanpassen om te zien hoe de reward verandert
TH_REF_FOR_PLOT = 0.0 # Bijvoorbeeld, je referentiehoek is 0 radialen
U_FOR_PLOT = 0.5      # Bijvoorbeeld, een constante control input van 0.5

# Bereken de beloningswaarden over het hele grid, met de gekozen vaste waarden
Z_REWARD = reward_function_for_plot(TH_MESH, OMEGA_MESH, th_ref_val=TH_REF_FOR_PLOT, u_val=U_FOR_PLOT)


plot_views = [
    {'elev': 30, 'azim': -60, 'title_suffix': 'Standaard Aanzicht'},
    {'elev': 90, 'azim': 90, 'title_suffix': 'Bovenaf (XY-vlak)'},
    {'elev': 0, 'azim': 0, 'title_suffix': 'Vooraanzicht (XZ-vlak)'},
    {'elev': 0, 'azim': 90, 'title_suffix': 'Zijaanzicht (YZ-vlak)'}
]

for i, view in enumerate(plot_views):
    fig = plt.figure(figsize=(12, 9))
    ax = fig.add_subplot(111, projection='3d')
    plt.tight_layout()

    ax.set_xlabel('x (rad)')
    ax.set_ylabel('y (rad/s)')
    ax.set_zlabel('z')
    ax.set_title(f'Visualisatie van z = f(x, y) - {view["title_suffix"]}\n(a={A_VALUE}, b={B_VALUE}, j={J_VALUE})')

    # --- Aangepaste x-as labels in veelvouden van pi ---
    # Bepaal de locaties van de ticks (bijv. -2pi, -pi, 0, pi, 2pi)
    x_tick_values = np.array([-2, -1.5, -1, -0.5, 0, 0.5, 1, 1.5, 2]) * np.pi
    ax.set_xticks(x_tick_values)
    
    # Maak de labels voor de ticks (bijv. "-2π", "-π", "0", "π", "2π")
    x_tick_labels = [f'${val}\pi$' if val != 0 else '0' for val in x_tick_values / np.pi]
    ax.set_xticklabels(x_tick_labels)
    # --- Einde aangepaste x-as labels ---

    # Plot het oppervlak
    surf = ax.plot_surface(TH_MESH, OMEGA_MESH, Z_REWARD, cmap='viridis', edgecolor='none', alpha=0.9)
    fig.colorbar(surf, shrink=0.5, aspect=5, label='Z Waarde')

    # Stel het aanzicht in
    ax.view_init(elev=view['elev'], azim=view['azim'])

    plt.show()


